# The Agentic Loop

In [1]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Say hello in one word"}],
)

print(response.content[0].text)

Hello


In [4]:
import json
MODEL = "claude-haiku-4-5-20251001"

weather_tool = {
    "name": "get_weather",
    "description": "Get the current weather in a given location",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City and state"}
        },
        "required": ["location"],
    },
}


def execute_tool(name, tool_input):
    if name == "get_weather":
        return f"Weather in {tool_input['location']}: 18°C, partly cloudy"
    return "Unknown tool"


messages = [{"role": "user", "content": "What's the weather in Toronto?"}]

# --- First call: Claude decides to use the tool ---
response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    tools=[weather_tool],
    messages=messages,
)

print("First response stop_reason:", response.stop_reason)  # -> "tool_use"
#print (response)


print(json.dumps(response.model_dump(), indent=2))

First response stop_reason: tool_use
{
  "id": "msg_011CeY4WrJa1q1bU8fa5HHf1",
  "container": null,
  "content": [
    {
      "id": "toolu_014ounejXMdqw4oj5TCdcM8L",
      "caller": {
        "type": "direct"
      },
      "input": {
        "location": "Toronto, Ontario"
      },
      "name": "get_weather",
      "type": "tool_use",
      "toolset_name": null
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "tool_use",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 574,
    "output_tokens": 56,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


In [ ]:
if response.stop_reason == "tool_use":
    # Find the tool_use block and run it
    tool_use_block = next(b for b in response.content if b.type == "tool_use")
    result = execute_tool(tool_use_block.name, tool_use_block.input)

    # Append Claude's tool call, then our tool result, to the conversation
    messages.append({"role": "assistant", "content": response.content})
    messages.append({
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": result,
            }
        ],
    })

    # --- Second call: Claude uses the tool result to give a final answer ---
    final_response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        tools=[weather_tool],
        messages=messages,
    )

    print("Final response stop_reason:", final_response.stop_reason)  # -> "end_turn"

    if final_response.stop_reason == "end_turn":
        for block in final_response.content:
            if block.type == "text":
                print("Claude says:", block.text)